In [1]:
### Load data
import pandas as pd   
# Prepare the historical panel (df1)
df1 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\B_Clustered_panel\03_Clustered_panel_1119.xlsx") 
df1 = df1.rename(columns={'Collaborative River Basin Planning': 'Collaborative Planning and Adaptation Strategies'})   # Rename the historical cooperation-mode column
df1.loc[df1['Both'] == 1, ['Conflict', 'Cooperation']] = 1
#df1['Cooperation'] = df1[label].any(axis=1).astype(int)   # Some panel rows have an active mode but Cooperation is not 1.
# Prepare future inputs (df2) and merge the mode indicators 
df2 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\E_ssp_predication\ssp1-5_future_data.xlsx", sheet_name='ssp1')  
df2 = df2[df2['year'] == 2050]  
# Prepare future cooperation-mode indicators (df3)
df3 = pd.read_csv(r"E:\AExperiment\11Water\E_RF_Data\D_strategy_onehot\20241218_updated_future_data_with_modes.csv")
df3.rename(columns={
    'ccode_iso3': 'country_code',
    'Collaborative River Basin Planning': 'Collaborative Planning and Adaptation Strategies'
}, inplace=True)   # Rename the country identifier and cooperation-mode column
df3 = df3[df3['year'] == 2050]   # Retain observations for 2050 only
# Define the selected columns
label = ['Joint Data-Sharing Systems','Cross-Border Basin Agreements','Transboundary Water Quality Standards',
         'Coordinated Hydropower Operations','Joint Water Allocation Models','Collaborative Planning and Adaptation Strategies'] 
# Select country_code and the mode-indicator columns from df3  
df3_labels = df3[['country_code'] + label]  
# Left-join df3_labels to df2  
df2 = pd.merge(df2, df3_labels, on='country_code', how='left')  
#df2.to_excel('D:\\OneDrive\\Desktop\\df2.xlsx', index=False)   # Optional intermediate output for inspection

In [2]:
### Combine historical and future inputs into df
import pandas as pd 
df2.rename(columns={  
    # Keep 'year' and 'export' unchanged when the column names already match 
    'country': 'Country',
    'country_code': 'ccode_iso3',
    'population_density_ssp1': 'population_density',  # Use the corresponding field names when switching SSP scenarios
    'GDP per capita_ssp1(constant 2015 US$)': 'gdp_percapita',  # Use the corresponding field names when switching SSP scenarios  
    'rural_population_ssp1(%)': 'rural_population',  # Use the corresponding field names when switching SSP scenarios
    'precipitation_ssp1': 'precipitation',   # Use the corresponding field names when switching SSP scenarios 
    'bws_ssp1': 'bws_score'}, inplace=True)   # Use the corresponding field names when switching SSP scenarios   
# Concatenate historical and future rows 
df = pd.concat([df1, df2], ignore_index=True, sort=True)
### Identify high-conflict countries for 2050 
df4 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\B_Clustered_panel\03_Clustered_panel_1119\Clustered_Conflict_Probability_241119.xlsx")
# Select rows assigned to Cluster 1  
high_conflict_country = df4[df4['Cluster'] == 1]  
# Extract the ISO3 country identifiers  
high_conflict_country = high_conflict_country['ccode_iso3'].tolist()  

# Select countries with high predicted conflict probabilities  
#high_conflict_country = ['LSO','LBN','SSD','SOM','ROU','USA','AFG','TUN','IRN','PSE','SAU','ZWE','MEX','LBY','LBR','PRY','QAT','AUT','SDN','ARE','TUR','MAR','TKM','RWA','MMR']

### Set Conflict to 1 for high-conflict countries in 2050
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Conflict'] = 1
#df.to_excel('D:\\OneDrive\\Desktop\\df.xlsx', index=False)   # Optional intermediate output for inspection

In [5]:
'''
### Carry the 2019 Average_conflict value forward to 2050
# Select observations for 2019  
df_2019_Average_conflict = df[df['year'] == 2019][['ccode_iso3', 'Average_conflict']]  
# Create a dictionary of the 2019 country-level values  
conflict_dict = df_2019_Average_conflict.set_index('ccode_iso3')['Average_conflict'].to_dict()  
# Update the 2050 observations  
df.loc[df['year'] == 2050, 'Average_conflict'] = df[df['year'] == 2050]['ccode_iso3'].map(conflict_dict)  
'''
### Project Average_conflict to 2050 using a time-series model
import pandas as pd  
from statsmodels.tsa.holtwinters import ExponentialSmoothing    
# Initialize a DataFrame for the 2050 forecasts  
predictions_2050 = pd.DataFrame(columns=['ccode_iso3', 'Average_conflict_2050'])  
# Forecast each country in high_conflict_country  
for country in high_conflict_country:  
    country_data = df[(df['ccode_iso3'] == country) & (df['year'].between(1951, 2019))]  
    if not country_data.empty:  
        print(f"Data for {country}:")  
        print(country_data[['Average_conflict']])  # Display the historical Average_conflict series for this country  
        # Fit exponential smoothing to the country-level time series  
        model = ExponentialSmoothing(country_data['Average_conflict'], trend='add', seasonal=None).fit()  
        # Forecast the series from 2020 through 2050  
        forecast = model.forecast(31)  # Forecast 31 steps beyond 2019 to reach 2050  
        print(f"Forecast for {country} from 2020 to 2050:")  
        print(forecast)  # Display the forecast  
        # Check that the forecast is available and whether its final value is negative  
        if not forecast.empty and forecast.iloc[-1] < 0:  
            forecast.iloc[-1] = 0  # Truncate a negative 2050 forecast to zero  
        # Check that a forecast is available after truncation  
        if not forecast.empty:  
            # Retain the forecast for 2050  
            forecast_2050 = forecast.iloc[-1]  
            new_prediction = pd.DataFrame({'ccode_iso3': [country], 'Average_conflict_2050': [forecast_2050]})  
            predictions_2050 = pd.concat([predictions_2050, new_prediction], ignore_index=True)  
        else:  
            print(f"No forecast available for {country}")  
# Merge the forecasts back into df  
df = df.merge(predictions_2050, on='ccode_iso3', how='left')  
# Update the projected values in the 2050 rows  
# Use the existing 2050 rows and update Average_conflict  
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Average_conflict'] = df['Average_conflict_2050']  
# Remove the temporary forecast column  
df.drop(columns=['Average_conflict_2050'], inplace=True)  
# Optionally inspect the updated DataFrame  
# print(df)  
# Save the results to an Excel file  
#df.to_excel('D:\\OneDrive\\Desktop\\time_series_2050.xlsx', index=False)

Data for AFG:
    Average_conflict
0           0.000000
1           0.000000
2           0.000000
3           0.000000
4           0.000000
..               ...
64          0.061538
65          0.060606
66          0.059701
67          0.073529
68          0.072464

[69 rows x 1 columns]
Forecast for AFG from 2020 to 2050:
69    0.075484
70    0.078168
71    0.080852
72    0.083535
73    0.086219
74    0.088903
75    0.091587
76    0.094270
77    0.096954
78    0.099638
79    0.102321
80    0.105005
81    0.107689
82    0.110373
83    0.113056
84    0.115740
85    0.118424
86    0.121107
87    0.123791
88    0.126475
89    0.129159
90    0.131842
91    0.134526
92    0.137210
93    0.139894
94    0.142577
95    0.145261
96    0.147945
97    0.150628
98    0.153312
99    0.155996
dtype: float64
Data for DZA:
     Average_conflict
142          0.000000
143          0.000000
144          0.000000
145          0.000000
146          0.000000
..                ...
206          0.153846
207  

D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1387: RuntimeWarning: divide by zero encountered in log
  bic = self.nobs * np.log(sse / self.nobs) + k * np.log(self.nobs)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1387: RuntimeWarning: divide by zero encountered in log
  bic = self.nobs * np.log(sse / self.nobs) + k * np.log(self.nobs)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimizat

Forecast for ETH from 2020 to 2050:
3686    0.162180
3687    0.164534
3688    0.166889
3689    0.169243
3690    0.171598
3691    0.173952
3692    0.176307
3693    0.178661
3694    0.181015
3695    0.183370
3696    0.185724
3697    0.188079
3698    0.190433
3699    0.192788
3700    0.195142
3701    0.197497
3702    0.199851
3703    0.202206
3704    0.204560
3705    0.206915
3706    0.209269
3707    0.211624
3708    0.213978
3709    0.216332
3710    0.218687
3711    0.221041
3712    0.223396
3713    0.225750
3714    0.228105
3715    0.230459
3716    0.232814
dtype: float64
Data for GIN:
      Average_conflict
4383          0.000000
4384          0.000000
4385          0.000000
4386          0.000000
4387          0.000000
...                ...
4447          0.015385
4448          0.015152
4449          0.074627
4450          0.073529
4451          0.072464

[69 rows x 1 columns]
Forecast for GIN from 2020 to 2050:
4452    0.090692
4453    0.104918
4454    0.119145
4455    0.133372
4456 

D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1387: RuntimeWarning: divide by zero encountered in log
  bic = self.nobs * np.log(sse / self.nobs) + k * np.log(self.nobs)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1387: RuntimeWarning: divide by zero encountered in log
  bic = self.nobs * np.log(sse / self.nobs) + k * np.log(self.nobs)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:903: ConvergenceWarning: Optimizat

Forecast for NGA from 2020 to 2050:
7783    0.176555
7784    0.179122
7785    0.181688
7786    0.184255
7787    0.186822
7788    0.189389
7789    0.191956
7790    0.194523
7791    0.197089
7792    0.199656
7793    0.202223
7794    0.204790
7795    0.207357
7796    0.209924
7797    0.212490
7798    0.215057
7799    0.217624
7800    0.220191
7801    0.222758
7802    0.225325
7803    0.227892
7804    0.230458
7805    0.233025
7806    0.235592
7807    0.238159
7808    0.240726
7809    0.243293
7810    0.245859
7811    0.248426
7812    0.250993
7813    0.253560
dtype: float64
Data for PSE:
      Average_conflict
8002          0.000000
8003          0.000000
8004          0.000000
8005          0.000000
8006          0.000000
...                ...
8066          0.015385
8067          0.015152
8068          0.029851
8069          0.000000
8070          0.000000

[69 rows x 1 columns]
Forecast for PSE from 2020 to 2050:
8071    0.001031
8072    0.001045
8073    0.001060
8074    0.001075
8075 

D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
D:\Anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self.

TypeError: Invalid value '[0.15599583161431102 0.3735269135150258 0.0 0.0 0.3378764388357355 0.0 0.0
 0.0 0.2328137028801811 0.5174969921907963 0.9804035803006661
 0.45164640520022103 0.3290999412207747 0.003047944513932049 0.0
 0.18928253494991926 0.1261298310694608 0.0 0.2535599434991713
 0.001474159466007659 0.06353095836132634 0.0 0.04220015615737421
 0.06338297568424683 0.0 0.47688521152583263 0.4076869915057918
 0.35869370697415726 0.4430945228756715 0.07256526412881642 0.0
 0.44641782666529106 0.3459026308144924]' for dtype 'float64'

In [ ]:
'''
###Replace the 2050 precipitation and bws_score inputs with their 2019 values
# Select the 2019 reference observations  
df_2019_prebws = df[df['year'] == 2019]  
# Iterate over the 2050 rows and replace the selected inputs  
for index, row in df[df['year'] == 2050].iterrows():  
    # Find the corresponding country observation for 2019  
    country_2019_data = df_2019_prebws[df_2019_prebws['ccode_iso3'] == row['ccode_iso3']]  
    if not country_2019_data.empty:  
        # Replace the selected input values  
        df.at[index, 'precipitation'] = country_2019_data['precipitation'].values[0]  
        df.at[index, 'bws_score'] = country_2019_data['bws_score'].values[0]  
# Inspect the DataFrame after replacement  
#df.to_excel('D:\\OneDrive\\Desktop\\output.xlsx', index=False)
#print(df)
'''

In [ ]:
### Preprocess the data
import numpy as np  
from sklearn.preprocessing import StandardScaler
# Create potential_Conflict as a copy of the original Conflict indicator  
#df['potential_Conflict'] = df['Conflict']   # The 2050 test sample retains conflict-coded observations, so potential_Conflict would equal 1.
# Define the selected columns
SEdata = ['population_density', 'export', 'gdp_percapita', 'rural_population', 'precipitation', 'bws_score']
# Select the analysis fields  
df = df[['year', 'ccode_iso3', 'Country_count', 'Average_conflict', 'Conflict'] + SEdata + label]
# Clean and standardize the data  
#pd.set_option('future.no_silent_downcasting', True)  
df = df.replace('..', np.nan)  
df = df.dropna(subset=df.columns.difference(['Country_count']))
# Standardize the selected continuous predictors
scaler = StandardScaler()  
df[SEdata] = scaler.fit_transform(df[SEdata])  
### Remove observations with missing inputs
#df = df.dropna(subset=df.columns.difference(['Country_count']))   # Drop rows missing any selected field other than Country_count.
# Display the number of retained observations
num_rows = df.shape[0]  
print("Number of rows:", num_rows)  
#df.to_excel('D:\\OneDrive\\Desktop\\output.xlsx', index=False)   # Optional intermediate output for inspection

In [ ]:
### Define the outcome and predictor variables
# Outcome and predictor variables  
y = 'Country_count'
x = df[['Average_conflict'] + ['Conflict'] + SEdata + label]
### Define the training and future prediction samples
# Select the training sample
mask_train = df['year'] <= 2019  
x_train = x[mask_train]  
y_train = df[y][mask_train]
# Select future observations after 2019  
# Select the future prediction sample  
mask_test = df['year'] > 2019
x_test = x[mask_test]  
y_test = df[y][mask_test]

In [ ]:
### Fit the Random Forest and compare scenario predictions
# Initialize the Random Forest regressor  
import pandas as pd  
from sklearn.ensemble import RandomForestRegressor  
rfr = RandomForestRegressor(random_state=42)  
rfr.fit(x_train, y_train)  
# Scenario 1 - Noncooperation: keep Conflict unchanged and set all mode indicators to 0  
x_test_case1 = x_test.copy()  
x_test_case1[label] = 0  
y_pred_case1 = rfr.predict(x_test_case1)  
# Store the predictions in new columns for the future sample  
df.loc[mask_test, 'Noncooperation'] = y_pred_case1  
# Scenario 2 - Cooperation: set Conflict to 0 and retain the mode indicators  
x_test_case2 = x_test.copy()  
x_test_case2['Conflict'] = 0  
y_pred_case2 = rfr.predict(x_test_case2)  
# Store the predictions in new columns for the future sample  
df.loc[mask_test, 'Cooperation'] = y_pred_case2  
# Scenario 3 - Mitigation: calculate Cooperation minus Noncooperation  
df.loc[mask_test, 'Mitigation'] = df.loc[mask_test, 'Cooperation'] - df.loc[mask_test, 'Noncooperation']  
### Calculate the means of Noncooperation, Cooperation, and Mitigation
mean_noncooperation = df.loc[mask_test, 'Noncooperation'].mean()
mean_cooperation = df.loc[mask_test, 'Cooperation'].mean()
mean_mitigation = df.loc[mask_test, 'Mitigation'].mean()
# Display the scenario results  
print(df[mask_test][['year', 'ccode_iso3', 'Noncooperation', 'Cooperation', 'Mitigation']])
print(f"Mean Non-Cooperation: {mean_noncooperation:.2f}\n"
      f"Mean Cooperation: {mean_cooperation:.2f}\n"
      f"Mean Mitigation: {mean_mitigation:.2f}")
# Export the results
df[mask_test][['year', 'ccode_iso3', 'Noncooperation', 'Cooperation', 'Mitigation']].to_excel('C:\\Users\\Lenovo\\Desktop\\ssp1_avecof_prebws.xlsx', index=False)